# 03 — Dedup Inspection

Inspect dedup statistics and spot-check near-duplicate clusters.

**Runtime:** CPU.

In [ ]:
import os, sys, json
os.chdir('/content/polygence'); sys.path.insert(0, 'src')
from prompt_classifier.config import load_config
cfg = load_config('configs/base.yaml')

with open(cfg['data']['stats_path']) as f:
    stats = json.load(f)
print('Dedup stats:')
for k in ['raw_total', 'dropped_length', 'dropped_exact_dedup', 'dropped_minhash_dedup', 'dropped_leak_check', 'total_after_dedup']:
    print(f'  {k}: {stats.get(k)}')

In [ ]:
# Spot-check: show jailbreakv examples that overlap with advbench/harmbench text
import pandas as pd
unified = pd.read_parquet(cfg['data']['unified_path'])
jv = unified[unified.source == 'jailbreakv'].head(5)
for _, row in jv.iterrows():
    print(f'[{row.source}] {row.prompt[:200]}\n')

In [ ]:
# Length outliers in block class
block = unified[unified.y == 1].copy()
block['char_len'] = block['prompt'].str.len()
long_prompts = block[block['char_len'] > 3000].sort_values('char_len', ascending=False)
print(f'{len(long_prompts)} block prompts > 3000 chars')
print(long_prompts[['source', 'char_len']].head(10))